# 04 公司基本信息数据清洗

## 目标

根据 `03_profile_audit.ipynb` 的审计结果，对公司基本信息训练数据进行正式清洗。

主要任务：

- 标准化股票代码；
- 去除公司名称首尾空格；
- 统一省级地区表示；
- 保持城市和行业的有效原始分类；
- 标准化所有制字段并保留真实缺失；
- 将多种上市日期格式统一为日期类型；
- 验证公司级主键唯一性；
- 验证与财务面板的匹配覆盖情况；
- 输出标准化 processed 数据。

原则：

- 不修改 raw 数据；
- 不机械统一合法的公司名称后缀；
- 不填补没有可靠依据的所有制和上市日期缺失；
- 每项转换先验证，再写回工作副本。

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

assert (PROJECT_ROOT / "pyproject.toml").exists()
assert DATA_RAW.exists()
assert DATA_PROCESSED.exists()

raw_profile = pd.read_csv(
    DATA_RAW / "firm_profile.csv",
    dtype={
        "stock_code": "string",
        "company_name": "string",
        "province": "string",
        "city": "string",
        "industry": "string",
        "ownership": "string",
    },
)

profile = raw_profile.copy()

print("raw shape:", raw_profile.shape)
print("working shape:", profile.shape)

profile.head()

raw shape: (42, 7)
working shape: (42, 7)


,stock_code,company_name,province,city,industry,ownership,listing_date
0,000001,华辰科技有限公司,广东,广州市,软件和信息技术服务业,国有,2005-01-01
1,2,新岳科技股份有限公司,广东省,深圳市,汽车制造业,民营,2006/02/02
2,3,海川科技股份有限公司,北京,北京市,医药制造业,外资,40034
3,000004,中盛科技股份有限公司,北京市,北京市,计算机通信和其他电子设备制造业,国有,NaN
4,000005,宏远科技股份有限公司,浙江,杭州市,专用设备制造业,民营,2009-05-05


In [2]:
stock_code_clean = (
    profile["stock_code"]
    .str.strip()
    .str.zfill(6)
)

stock_code_changed_mask = (
    profile["stock_code"]
    .ne(stock_code_clean)
    .fillna(False)
)

stock_code_changes = pd.DataFrame(
    {
        "before": profile.loc[
            stock_code_changed_mask,
            "stock_code",
        ].map(repr),
        "after": stock_code_clean.loc[
            stock_code_changed_mask
        ].map(repr),
    }
)

print(
    "发生标准化变化的股票代码记录数:",
    stock_code_changed_mask.sum(),
)

stock_code_changes

发生标准化变化的股票代码记录数: 12


,before,after
1,'2','000002'
2,'3','000003'
3,' 000004 ','000004'
11,'12','000012'
12,'13','000013'
13,' 000014 ','000014'
21,'22','000022'
22,'23','000023'
23,' 000024 ','000024'
31,'32','000032'


In [3]:
print(
    "清洗后缺失数:",
    stock_code_clean.isna().sum(),
)

print(
    "清洗后非6位数字代码数:",
    (
        stock_code_clean.notna()
        & ~stock_code_clean.str.fullmatch(r"\d{6}")
    ).sum(),
)

print(
    "清洗后唯一股票代码数:",
    stock_code_clean.nunique(),
)

清洗后缺失数: 0
清洗后非6位数字代码数: 0
清洗后唯一股票代码数: 42


In [4]:
assert stock_code_clean.isna().sum() == 0

assert (
    stock_code_clean
    .str.fullmatch(r"\d{6}")
    .all()
)

assert stock_code_clean.nunique() == 42

profile["stock_code"] = stock_code_clean

print("stock_code 标准化完成")

stock_code 标准化完成


In [5]:
company_name_clean = (
    profile["company_name"]
    .str.strip()
)

company_name_changed_mask = (
    profile["company_name"]
    .ne(company_name_clean)
    .fillna(False)
)

company_name_changes = pd.DataFrame(
    {
        "before": profile.loc[
            company_name_changed_mask,
            "company_name",
        ].map(repr),
        "after": company_name_clean.loc[
            company_name_changed_mask
        ].map(repr),
    }
)

print(
    "公司名称发生变化的记录数:",
    company_name_changed_mask.sum(),
)

company_name_changes

公司名称发生变化的记录数: 3


,before,after
13,' 明德科技股份有限公司 ','明德科技股份有限公司'
26,' 兆丰科技股份有限公司 ','兆丰科技股份有限公司'
39,' 鸿远科技股份有限公司 ','鸿远科技股份有限公司'


In [6]:
assert company_name_clean.isna().sum() == 0
assert company_name_clean.nunique() == 42

profile["company_name"] = company_name_clean

print("company_name 首尾空格清理完成")

company_name 首尾空格清理完成


In [7]:
province_map = {
    "上海": "上海市",
    "上海市": "上海市",
    "北京": "北京市",
    "北京市": "北京市",
    "广东": "广东省",
    "广东省": "广东省",
    "江苏": "江苏省",
    "江苏省": "江苏省",
    "浙江": "浙江省",
    "浙江省": "浙江省",
    "四川省": "四川省",
    "湖北省": "湖北省",
}

province_before = (
    profile["province"]
    .astype("string")
    .str.strip()
)

province_clean = (
    province_before
    .map(province_map)
    .astype("string")
)

province_changed_mask = (
    province_before
    .ne(province_clean)
    .fillna(False)
)

province_changes = pd.DataFrame(
    {
        "before": province_before.loc[
            province_changed_mask
        ],
        "after": province_clean.loc[
            province_changed_mask
        ],
    }
)

print(
    "省份发生标准化变化的记录数:",
    province_changed_mask.sum(),
)

print(
    "标准化后缺失数:",
    province_clean.isna().sum(),
)

print(
    "标准化后省份类别数:",
    province_clean.nunique(),
)

province_changes

省份发生标准化变化的记录数: 17
标准化后缺失数: 0
标准化后省份类别数: 7


,before,after
0,广东,广东省
2,北京,北京市
4,浙江,浙江省
6,江苏,江苏省
8,上海,上海市
12,广东,广东省
14,北京,北京市
16,浙江,浙江省
18,江苏,江苏省
20,上海,上海市


In [8]:
print(
    province_clean
    .value_counts()
    .sort_index()
)

province
上海市    6
北京市    8
四川省    3
广东省    9
江苏省    6
浙江省    7
湖北省    3
Name: count, dtype: int64[pyarrow]


In [9]:
assert province_clean.isna().sum() == 0
assert province_clean.nunique() == 7

profile["province"] = province_clean

print("province 标准化完成")

province 标准化完成


In [10]:
city_clean = (
    profile["city"]
    .astype("string")
    .str.strip()
)

city_changed_mask = (
    profile["city"]
    .astype("string")
    .ne(city_clean)
    .fillna(False)
)

print(
    "城市发生变化的记录数:",
    city_changed_mask.sum(),
)

print(
    "城市缺失数:",
    city_clean.isna().sum(),
)

print(
    "城市类别数:",
    city_clean.nunique(),
)

print(
    "不以“市”结尾的记录数:",
    (
        city_clean.notna()
        & ~city_clean.str.endswith("市")
    ).sum(),
)

城市发生变化的记录数: 0
城市缺失数: 0
城市类别数: 12
不以“市”结尾的记录数: 0


In [11]:
assert city_clean.isna().sum() == 0

assert (
    city_clean.notna()
    & ~city_clean.str.endswith("市")
).sum() == 0

profile["city"] = city_clean

print("city 标准化完成")

city 标准化完成


In [12]:
industry_clean = (
    profile["industry"]
    .astype("string")
    .str.strip()
)

industry_changed_mask = (
    profile["industry"]
    .astype("string")
    .ne(industry_clean)
    .fillna(False)
)

print(
    "行业发生变化的记录数:",
    industry_changed_mask.sum(),
)

print(
    "行业缺失数:",
    industry_clean.isna().sum(),
)

print(
    "行业类别数:",
    industry_clean.nunique(),
)

print(
    industry_clean
    .value_counts()
    .sort_index()
)

行业发生变化的记录数: 0
行业缺失数: 0
行业类别数: 6
industry
专用设备制造业            7
医药制造业              7
汽车制造业              7
电气机械和器材制造业         6
计算机通信和其他电子设备制造业    7
软件和信息技术服务业         8
Name: count, dtype: int64[pyarrow]


In [13]:
assert industry_clean.isna().sum() == 0
assert industry_clean.nunique() == 6

profile["industry"] = industry_clean

print("industry 标准化完成")

industry 标准化完成


In [14]:
ownership_clean = (
    profile["ownership"]
    .astype("string")
    .str.strip()
)

ownership_changed_mask = (
    profile["ownership"]
    .astype("string")
    .ne(ownership_clean)
    .fillna(False)
)

print(
    "ownership 发生格式变化的记录数:",
    ownership_changed_mask.sum(),
)

print(
    "ownership 缺失数:",
    ownership_clean.isna().sum(),
)

print(
    "ownership 非缺失类别数:",
    ownership_clean.nunique(
        dropna=True
    ),
)

print(
    ownership_clean
    .value_counts(
        dropna=False
    )
    .sort_index()
)

ownership 发生格式变化的记录数: 0
ownership 缺失数: 3
ownership 非缺失类别数: 3
ownership
国有      14
外资      12
民营      13
<NA>     3
Name: count, dtype: int64[pyarrow]


In [15]:
valid_ownership = {
    "国有",
    "民营",
    "外资",
}

invalid_ownership_mask = (
    ownership_clean.notna()
    & ~ownership_clean.isin(
        valid_ownership
    )
)

print(
    "未知 ownership 类别记录数:",
    invalid_ownership_mask.sum(),
)

未知 ownership 类别记录数: 0


In [16]:
assert ownership_clean.isna().sum() == 3

assert (
    ownership_clean.dropna()
    .isin(valid_ownership)
    .all()
)

profile["ownership"] = ownership_clean

print("ownership 标准化完成")

ownership 标准化完成


In [17]:
profile_stage_summary = pd.DataFrame(
    {
        "dtype": profile.dtypes.astype(str),
        "missing": profile.isna().sum(),
        "unique": profile.nunique(
            dropna=True
        ),
    }
)

profile_stage_summary

,dtype,missing,unique
stock_code,string,0,42
company_name,string,0,42
province,string,0,7
city,string,0,12
industry,string,0,6
ownership,string,3,3
listing_date,str,10,32


In [18]:
print(
    "当前记录数:",
    len(profile),
)

print(
    "唯一股票代码数:",
    profile["stock_code"].nunique(),
)

print(
    "重复 stock_code 记录数:",
    profile
    .duplicated(
        subset=["stock_code"],
        keep=False,
    )
    .sum(),
)

当前记录数: 42
唯一股票代码数: 42
重复 stock_code 记录数: 0


In [19]:
listing_text = (
    profile["listing_date"]
    .astype("string")
    .str.strip()
)

listing_dash_mask = (
    listing_text.str.fullmatch(
        r"\d{4}-\d{2}-\d{2}",
        na=False,
    )
)

listing_slash_mask = (
    listing_text.str.fullmatch(
        r"\d{4}/\d{2}/\d{2}",
        na=False,
    )
)

listing_excel_serial_mask = (
    listing_text.str.fullmatch(
        r"\d+",
        na=False,
    )
)

listing_missing_mask = (
    listing_text.isna()
)

listing_unknown_mask = ~(
    listing_dash_mask
    | listing_slash_mask
    | listing_excel_serial_mask
    | listing_missing_mask
)

print("YYYY-MM-DD:", listing_dash_mask.sum())
print("YYYY/MM/DD:", listing_slash_mask.sum())
print(
    "Excel 序列号:",
    listing_excel_serial_mask.sum(),
)
print("原始缺失:", listing_missing_mask.sum())
print("未知格式:", listing_unknown_mask.sum())

YYYY-MM-DD: 11
YYYY/MM/DD: 11
Excel 序列号: 10
原始缺失: 10
未知格式: 0


In [20]:
listing_date_clean = pd.Series(
    pd.NaT,
    index=profile.index,
    dtype="datetime64[ns]",
)

listing_date_clean.loc[
    listing_dash_mask
] = pd.to_datetime(
    listing_text.loc[
        listing_dash_mask
    ],
    format="%Y-%m-%d",
)

listing_date_clean.loc[
    listing_slash_mask
] = pd.to_datetime(
    listing_text.loc[
        listing_slash_mask
    ],
    format="%Y/%m/%d",
)

excel_serial_values = pd.to_numeric(
    listing_text.loc[
        listing_excel_serial_mask
    ],
    errors="raise",
)

listing_date_clean.loc[
    listing_excel_serial_mask
] = pd.to_datetime(
    excel_serial_values,
    unit="D",
    origin="1899-12-30",
)

In [21]:
listing_date_validation = pd.Series(
    {
        "raw_non_missing": (
            listing_text.notna().sum()
        ),
        "clean_non_missing": (
            listing_date_clean.notna().sum()
        ),
        "raw_missing": (
            listing_text.isna().sum()
        ),
        "clean_missing": (
            listing_date_clean.isna().sum()
        ),
        "unknown_format": (
            listing_unknown_mask.sum()
        ),
    }
)

listing_date_validation

raw_non_missing      32
clean_non_missing    32
raw_missing          10
clean_missing        10
unknown_format        0
dtype: int64

In [22]:
print(
    "最早上市日期:",
    listing_date_clean.min(),
)

print(
    "最晚上市日期:",
    listing_date_clean.max(),
)

print(
    "日期 dtype:",
    listing_date_clean.dtype,
)

最早上市日期: 2005-01-01 00:00:00
最晚上市日期: 2019-09-20 00:00:00
日期 dtype: datetime64[ns]


In [23]:
listing_date_changes = pd.DataFrame(
    {
        "before": listing_text,
        "after": listing_date_clean,
    }
)

listing_date_changes.loc[
    (
        listing_slash_mask
        | listing_excel_serial_mask
    )
].head(15)

,before,after
1,2006/02/02,2006-02-02
2,40034,2009-08-09
5,2010/06/06,2010-06-06
6,40102,2009-10-16
9,2014/10/10,2014-10-10
10,40170,2009-12-23
13,2018/02/14,2018-02-14
14,40238,2010-03-01
17,2007/06/18,2007-06-18
18,40306,2010-05-08


In [24]:
assert listing_unknown_mask.sum() == 0

assert (
    listing_text.notna().sum()
    == listing_date_clean.notna().sum()
)

assert listing_date_clean.isna().sum() == 10

profile["listing_date"] = listing_date_clean

print("listing_date 标准化完成")
print(
    "listing_date dtype:",
    profile["listing_date"].dtype,
)

listing_date 标准化完成
listing_date dtype: datetime64[ns]


In [25]:
profile_cleaning_summary = pd.DataFrame(
    {
        "dtype": profile.dtypes.astype(str),
        "missing": profile.isna().sum(),
        "unique": profile.nunique(
            dropna=True
        ),
    }
)

profile_cleaning_summary

,dtype,missing,unique
stock_code,string,0,42
company_name,string,0,42
province,string,0,7
city,string,0,12
industry,string,0,6
ownership,string,3,3
listing_date,datetime64[ns],10,31


In [26]:
duplicate_listing_date_mask = (
    profile["listing_date"]
    .notna()
    & profile["listing_date"]
    .duplicated(keep=False)
)

duplicate_listing_dates = (
    profile.loc[
        duplicate_listing_date_mask,
        [
            "stock_code",
            "company_name",
            "listing_date",
        ],
    ]
    .sort_values(
        [
            "listing_date",
            "stock_code",
        ]
    )
)

print(
    "处于重复上市日期中的公司记录数:",
    len(duplicate_listing_dates),
)

print(
    "重复上市日期的日期种类数:",
    duplicate_listing_dates[
        "listing_date"
    ].nunique(),
)

duplicate_listing_dates

处于重复上市日期中的公司记录数: 2
重复上市日期的日期种类数: 1


,stock_code,company_name,listing_date
20,000021,恒通科技股份有限公司,2010-09-21
26,000027,兆丰科技股份有限公司,2010-09-21


In [27]:
print(
    "总记录数:",
    len(profile),
)

print(
    "唯一 stock_code 数:",
    profile["stock_code"].nunique(),
)

print(
    "重复 stock_code 记录数:",
    profile
    .duplicated(
        subset=["stock_code"],
        keep=False,
    )
    .sum(),
)

print(
    "完全重复记录数:",
    profile
    .duplicated(keep=False)
    .sum(),
)

总记录数: 42
唯一 stock_code 数: 42
重复 stock_code 记录数: 0
完全重复记录数: 0


In [28]:
financials_clean = pd.read_parquet(
    DATA_PROCESSED
    / "firm_financials_clean.parquet"
)

financial_codes = set(
    financials_clean[
        "stock_code"
    ].astype("string")
)

profile_codes = set(
    profile[
        "stock_code"
    ].astype("string")
)

financial_missing_profile = (
    financial_codes
    - profile_codes
)

profile_extra_codes = (
    profile_codes
    - financial_codes
)

print(
    "财务公司数:",
    len(financial_codes),
)

print(
    "Profile 公司数:",
    len(profile_codes),
)

print(
    "财务中无法匹配 Profile 的公司数:",
    len(financial_missing_profile),
)

print(
    "Profile 中财务样本外公司数:",
    len(profile_extra_codes),
)

print(
    "Profile 样本外公司:",
    sorted(profile_extra_codes),
)

财务公司数: 40
Profile 公司数: 42
财务中无法匹配 Profile 的公司数: 0
Profile 中财务样本外公司数: 2
Profile 样本外公司: ['900001', '900002']


In [29]:
profile_merge_check = (
    financials_clean[
        [
            "stock_code",
            "year",
        ]
    ]
    .merge(
        profile[
            [
                "stock_code",
                "company_name",
                "province",
                "city",
                "industry",
                "ownership",
                "listing_date",
            ]
        ],
        on="stock_code",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
)

print(
    "合并后记录数:",
    len(profile_merge_check),
)

print(
    "\n匹配状态:"
)

print(
    profile_merge_check[
        "_merge"
    ].value_counts()
)

合并后记录数: 240

匹配状态:
_merge
both          240
left_only       0
right_only      0
Name: count, dtype: int64


In [30]:
profile_final_summary = pd.DataFrame(
    {
        "dtype": profile.dtypes.astype(str),
        "missing": profile.isna().sum(),
        "unique": profile.nunique(
            dropna=True
        ),
    }
)

profile_final_summary

,dtype,missing,unique
stock_code,string,0,42
company_name,string,0,42
province,string,0,7
city,string,0,12
industry,string,0,6
ownership,string,3,3
listing_date,datetime64[ns],10,31


## 第五阶段：最终验收与 processed 数据输出

字段级清洗完成后，对公司基本信息数据进行最终验收。

重点验证：

- `stock_code` 为 6 位字符串且公司级唯一；
- 公司名称无首尾空格；
- 省级地区表示统一；
- 城市和行业分类保持有效；
- `ownership` 仅保留原始真实缺失；
- `listing_date` 已统一为日期类型；
- 公司基本信息能够覆盖财务面板全部公司；
- Profile 中额外公司不会扩充财务研究样本。

验收后分别输出 Parquet 和 Stata `.dta` 版本。

In [31]:
expected_provinces = {
    "上海市",
    "北京市",
    "四川省",
    "广东省",
    "江苏省",
    "浙江省",
    "湖北省",
}

valid_ownership = {
    "国有",
    "民营",
    "外资",
}

assert len(profile) == 42

assert profile["stock_code"].isna().sum() == 0
assert profile["stock_code"].nunique() == 42
assert profile["stock_code"].str.fullmatch(r"\d{6}").all()

assert (
    profile["company_name"]
    .eq(profile["company_name"].str.strip())
    .all()
)

assert set(profile["province"].dropna()) == expected_provinces

assert profile["city"].isna().sum() == 0
assert profile["city"].str.endswith("市").all()

assert profile["industry"].isna().sum() == 0
assert profile["industry"].nunique() == 6

assert profile["ownership"].isna().sum() == 3
assert (
    profile["ownership"]
    .dropna()
    .isin(valid_ownership)
    .all()
)

assert profile["listing_date"].isna().sum() == 10
assert profile["listing_date"].notna().sum() == 32

assert (
    profile
    .duplicated(
        subset=["stock_code"],
        keep=False,
    )
    .sum()
    == 0
)

assert len(financial_missing_profile) == 0
assert profile_extra_codes == {
    "900001",
    "900002",
}

assert len(profile_merge_check) == 240
assert (
    profile_merge_check["_merge"]
    .eq("both")
    .all()
)

print("Profile 输出前质量验收通过")
print("公司记录数:", len(profile))
print("唯一 stock_code:", profile["stock_code"].nunique())
print("财务面板匹配 firm-year:", len(profile_merge_check))

Profile 输出前质量验收通过
公司记录数: 42
唯一 stock_code: 42
财务面板匹配 firm-year: 240


In [32]:
profile_final = (
    profile
    .sort_values("stock_code")
    .reset_index(drop=True)
    .copy()
)

profile_final.head()

,stock_code,company_name,province,city,industry,ownership,listing_date
0,000001,华辰科技有限公司,广东省,广州市,软件和信息技术服务业,国有,2005-01-01
1,000002,新岳科技股份有限公司,广东省,深圳市,汽车制造业,民营,2006-02-02
2,000003,海川科技股份有限公司,北京市,北京市,医药制造业,外资,2009-08-09
3,000004,中盛科技股份有限公司,北京市,北京市,计算机通信和其他电子设备制造业,国有,NaT
4,000005,宏远科技股份有限公司,浙江省,杭州市,专用设备制造业,民营,2009-05-05


In [33]:
DATA_PROCESSED.mkdir(
    parents=True,
    exist_ok=True,
)

profile_parquet_path = (
    DATA_PROCESSED
    / "firm_profile_clean.parquet"
)

profile_final.to_parquet(
    profile_parquet_path,
    index=False,
)

print(
    "Parquet 输出:",
    profile_parquet_path.name,
)

Parquet 输出: firm_profile_clean.parquet


In [34]:
profile_stata = (
    profile_final.copy()
)

stata_string_cols = [
    "stock_code",
    "company_name",
    "province",
    "city",
    "industry",
    "ownership",
]

for col in stata_string_cols:
    profile_stata[col] = (
        profile_stata[col]
        .astype(object)
    )

    profile_stata.loc[
        profile_stata[col].isna(),
        col,
    ] = None

In [35]:
print(
    "Stata 导出前 ownership 缺失:",
    profile_stata["ownership"].isna().sum(),
)

print(
    "Stata 导出前 listing_date 缺失:",
    profile_stata["listing_date"].isna().sum(),
)

Stata 导出前 ownership 缺失: 3
Stata 导出前 listing_date 缺失: 10


In [36]:
profile_stata_path = (
    DATA_PROCESSED
    / "firm_profile_clean.dta"
)

profile_stata.to_stata(
    profile_stata_path,
    write_index=False,
    version=118,
    convert_dates={
        "listing_date": "td",
    },
)

print(
    "Stata 输出:",
    profile_stata_path.name,
)

Stata 输出: firm_profile_clean.dta


In [37]:
profile_parquet_check = pd.read_parquet(
    profile_parquet_path
)

assert len(profile_parquet_check) == 42

assert (
    profile_parquet_check[
        "stock_code"
    ]
    .str.fullmatch(r"\d{6}")
    .all()
)

assert (
    profile_parquet_check
    .duplicated(
        subset=["stock_code"],
        keep=False,
    )
    .sum()
    == 0
)

assert (
    profile_parquet_check[
        "ownership"
    ].isna().sum()
    == 3
)

assert (
    profile_parquet_check[
        "listing_date"
    ].isna().sum()
    == 10
)

print("Profile Parquet 回读验证通过")
print(
    "shape:",
    profile_parquet_check.shape,
)

profile_parquet_check.dtypes

Profile Parquet 回读验证通过
shape: (42, 7)


stock_code              string
company_name            string
province                string
city                    string
industry                string
ownership               string
listing_date    datetime64[ns]
dtype: object

In [38]:
profile_stata_check = pd.read_stata(
    profile_stata_path,
    convert_categoricals=False,
)

print(
    "shape:",
    profile_stata_check.shape,
)

profile_stata_check.dtypes

shape: (42, 7)


stock_code                str
company_name              str
province                  str
city                      str
industry                  str
ownership                 str
listing_date    datetime64[s]
dtype: object

In [39]:
stata_ownership_check = (
    profile_stata_check[
        "ownership"
    ]
    .replace("", pd.NA)
    .astype("string")
)

assert len(profile_stata_check) == 42

assert (
    profile_stata_check[
        "stock_code"
    ]
    .str.fullmatch(r"\d{6}")
    .all()
)

assert (
    profile_stata_check[
        "stock_code"
    ].nunique()
    == 42
)

assert (
    stata_ownership_check
    .isna()
    .sum()
    == 3
)

assert (
    profile_stata_check[
        "listing_date"
    ]
    .isna()
    .sum()
    == 10
)

print(
    "Profile Stata .dta 回读验证通过"
)

Profile Stata .dta 回读验证通过


In [40]:
parquet_profile_codes = set(
    profile_parquet_check[
        "stock_code"
    ]
)

stata_profile_codes = set(
    profile_stata_check[
        "stock_code"
    ]
)

print(
    "Parquet 公司数:",
    len(parquet_profile_codes),
)

print(
    "Stata 公司数:",
    len(stata_profile_codes),
)

print(
    "两种格式股票代码完全一致:",
    (
        parquet_profile_codes
        == stata_profile_codes
    ),
)

Parquet 公司数: 42
Stata 公司数: 42
两种格式股票代码完全一致: True


## 本阶段最终结论

本 Notebook 已完成模拟公司基本信息数据的正式清洗和验证。

主要处理结果：

- 原始数据共 42 条公司级记录、7 个字段；
- 12 条股票代码经过前导零和首尾空格标准化，最终全部为 6 位字符串；
- 标准化后仍保持 42 个唯一股票代码，无代码碰撞；
- 3 条公司名称首尾空格被清理，合法公司组织形式保持原样；
- 省级地区由 12 种原始表示统一为 7 个正式省级地区，共 17 条记录发生标准化；
- 城市字段无需额外修改，共 12 类；
- 行业字段保持原始 6 类有效分类；
- `ownership` 保留 3 条原始真实缺失，不进行无依据填补；
- `listing_date` 中 11 条 `YYYY-MM-DD`、11 条 `YYYY/MM/DD` 和 10 条 Excel 日期序列号均成功统一为日期类型；
- 上市日期保留 10 条原始缺失，没有因转换制造新缺失；
- 32 条非缺失上市日期对应 31 个唯一日期，其中两家公司合法共享同一上市日期；
- 公司基本信息能够覆盖财务面板全部 40 家公司；
- Profile 中另外存在 `900001`、`900002` 两家财务样本外公司，不用于扩充财务研究样本。

清洗结果已分别输出为 Parquet 和 Stata `.dta` 格式，并完成：

- Parquet 写入和回读验证；
- Stata `.dta` 写入和 Python 回读验证；
- 两种格式股票代码集合一致性验证；
- Stata/MP 实际加载验证；
- Stata `isid stock_code` 公司级主键唯一性验证；
- 所有制和上市日期缺失结构验证。

原始 raw 数据始终未被修改。

下一阶段将进入专利数据的审计和清洗，并逐步构建公司年度研究面板。